In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
# ===================== NB3 CELL 1 - Setup (GPU T4, Internet On, secret "HF") =====================
!pip install -q bert-score

import os, sys, json, time, shutil
os.environ["HF_HOME"] = "/tmp/hf_home"
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi, hf_hub_download, snapshot_download
HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)
HF_USER   = api.whoami()["name"]
DATA_REPO = f"{HF_USER}/cs781-p1-data"
CKPT_REPO = f"{HF_USER}/cs781-p1-ckpt"
SRC_REPO  = "Exploration-Lab/CS-781-Capstone"
WORK      = "/kaggle/working"
DATA_DIR  = "/tmp/cs781/prepared"
CKPT_DIR  = "/tmp/cs781/ckpt"
RESULTS   = f"{WORK}/results"; os.makedirs(RESULTS, exist_ok=True)

snapshot_download(DATA_REPO, repo_type="dataset", local_dir=DATA_DIR, token=HF_TOKEN,
                  allow_patterns=["tokenizer.json", "dev.json", "dev_truth.json", "meta.json"])
test_path = hf_hub_download(SRC_REPO, "Phase_1_Pretraining/Test/test.json", repo_type="dataset", token=HF_TOKEN,
                            local_dir="/tmp/cs781/raw")

import torch
from tokenizers import Tokenizer
tok = Tokenizer.from_file(f"{DATA_DIR}/tokenizer.json")
dev_prompts = json.load(open(f"{DATA_DIR}/dev.json"))
dev_truth   = json.load(open(f"{DATA_DIR}/dev_truth.json"))
DEV_IDS     = list(dev_prompts)
test_data   = json.load(open(test_path))
TEST_IDS    = list(test_data)
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device {DEVICE} | dev {len(DEV_IDS)} | test {len(TEST_IDS)} | torch {torch.__version__}")
print("checkpoints on HF:", sorted({f.split('/')[1] for f in api.list_repo_files(CKPT_REPO) if f.startswith('runs/')}))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.4 MB/s eta 0:00:00ta 0:00:01


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]

device cuda | dev 479 | test 1000 | torch 2.11.0+cu128
checkpoints on HF: ['explore_encdec', 'explore_gpt2', 'explore_llama', 'explore_shared', 'main_llama', 'smoke_llama']


In [3]:
%%writefile /kaggle/working/model.py
"""CS-781 Phase 1 - four legal language-model architectures, written from scratch in PyTorch.

  gpt2    : classic decoder-only Transformer (learned absolute positions, LayerNorm, GELU MLP, multi-head attention)
  llama   : modern decoder-only Transformer (RoPE, RMSNorm, SwiGLU, grouped-query attention, no biases)
  shared  : lightweight decoder (ALBERT-style factorised embeddings + cycle cross-layer weight sharing + multi-query attn)
  encdec  : encoder-decoder Transformer (bidirectional encoder over the prompt, causal decoder with cross-attention)

All models: tied input/output embeddings, pre-norm residual blocks, PyTorch SDPA attention, KV-cache generation.
"""
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class ModelConfig:
    arch: str = "llama"
    vocab_size: int = 32768
    d_model: int = 768
    n_layer: int = 12            # decoder-only: layers; encdec: layers in EACH of encoder and decoder; shared: effective depth
    n_head: int = 12
    n_kv_head: int = 4           # GQA groups (llama/encdec); shared uses 1 (MQA); gpt2 uses n_head (MHA)
    d_ff: int = 2304
    max_seq_len: int = 1024
    rope_theta: float = 10000.0
    dropout: float = 0.0
    n_unique_layers: int = 6     # shared only: number of distinct blocks, reused cyclically
    d_embed: int = 256           # shared only: factorised embedding size
    norm_eps: float = 1e-5

    def to_dict(self):
        return asdict(self)


def preset(arch: str, **overrides) -> ModelConfig:
    base = {
        "gpt2":   dict(arch="gpt2", d_model=768, n_layer=12, n_head=12, n_kv_head=12, d_ff=3072),
        "llama":  dict(arch="llama", d_model=768, n_layer=12, n_head=12, n_kv_head=4, d_ff=2304),
        "shared": dict(arch="shared", d_model=768, n_layer=12, n_head=12, n_kv_head=1, d_ff=3072,
                       n_unique_layers=6, d_embed=256),
        "encdec": dict(arch="encdec", d_model=768, n_layer=6, n_head=12, n_kv_head=4, d_ff=2304),
    }[arch]
    base.update(overrides)
    return ModelConfig(**base)


# ----------------------------------------------------------------------------------------------------------------
# building blocks
# ----------------------------------------------------------------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).to(x.dtype)


class LayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.ln = nn.LayerNorm(d, eps=eps)

    def forward(self, x):
        return F.layer_norm(x.float(), self.ln.normalized_shape, self.ln.weight.float(), self.ln.bias.float(),
                            self.ln.eps).to(x.dtype)


def rope_cache(head_dim, max_len, theta, device=None):
    inv = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(max_len, device=device).float()
    freqs = torch.outer(t, inv)                       # (T, hd/2)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, cos, sin):
    # x: (B, H, T, hd); cos/sin: (T, hd/2) - rotate pairs (x[..., :hd/2], x[..., hd/2:])
    xf = x.float()
    x1, x2 = xf[..., : xf.shape[-1] // 2], xf[..., xf.shape[-1] // 2:]
    cos, sin = cos[None, None], sin[None, None]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.to(x.dtype)


class Attention(nn.Module):
    """Multi-head / grouped-query / multi-query attention. Self-attention (optionally causal) or cross-attention."""

    def __init__(self, cfg: ModelConfig, n_kv_head: int, bias: bool, use_rope: bool, cross: bool = False):
        super().__init__()
        d, h = cfg.d_model, cfg.n_head
        assert d % h == 0 and h % n_kv_head == 0
        self.h, self.kv, self.hd = h, n_kv_head, d // h
        self.use_rope, self.cross = use_rope, cross
        self.q = nn.Linear(d, h * self.hd, bias=bias)
        self.k = nn.Linear(d, n_kv_head * self.hd, bias=bias)
        self.v = nn.Linear(d, n_kv_head * self.hd, bias=bias)
        self.o = nn.Linear(h * self.hd, d, bias=bias)
        self.o.is_residual_proj = True
        self.dropout = cfg.dropout

    def _split(self, x, n):
        B, T, _ = x.shape
        return x.view(B, T, n, self.hd).transpose(1, 2)          # (B, n, T, hd)

    def _expand(self, x):
        return x if self.kv == self.h else x.repeat_interleave(self.h // self.kv, dim=1)

    def forward(self, x, rope=None, pos=0, causal=True, cache=None, memory=None):
        """
        x: (B, T, d).  rope: (cos, sin) full tables.  pos: absolute position of x[:,0] (for RoPE during decoding).
        cache: dict used during generation. Self-attn: keys 'k','v' accumulate. Cross-attn: 'k','v' computed once.
        memory: encoder output (B, S, d) for cross-attention.
        """
        q = self._split(self.q(x), self.h)
        if self.cross:
            if cache is not None and "k" in cache:
                k, v = cache["k"], cache["v"]
            else:
                k, v = self._split(self.k(memory), self.kv), self._split(self.v(memory), self.kv)
                if cache is not None:
                    cache["k"], cache["v"] = k, v
            is_causal = False
        else:
            k, v = self._split(self.k(x), self.kv), self._split(self.v(x), self.kv)
            if self.use_rope:
                cos, sin = rope
                T = x.shape[1]
                q = apply_rope(q, cos[pos:pos + T], sin[pos:pos + T])
                k = apply_rope(k, cos[pos:pos + T], sin[pos:pos + T])
            if cache is not None:
                if "k" in cache:
                    k = torch.cat([cache["k"], k], dim=2); v = torch.cat([cache["v"], v], dim=2)
                cache["k"], cache["v"] = k, v
            # causal mask only needed when queries and keys cover the same span (training / prefill)
            is_causal = causal and q.shape[2] == k.shape[2] and q.shape[2] > 1
            if causal and q.shape[2] != k.shape[2] and q.shape[2] != 1:
                raise ValueError("multi-token decode with a non-empty cache is not supported")
        y = F.scaled_dot_product_attention(q, self._expand(k), self._expand(v), is_causal=is_causal,
                                           dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(x.shape[0], x.shape[1], self.h * self.hd)
        return self.o(y)


class MLP(nn.Module):
    def __init__(self, cfg: ModelConfig, kind: str, bias: bool):
        super().__init__()
        self.kind = kind
        if kind == "swiglu":
            self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
            self.up = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
        else:
            self.up = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=bias)
        self.down.is_residual_proj = True
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        if self.kind == "swiglu":
            h = F.silu(self.gate(x)) * self.up(x)
        else:
            h = F.gelu(self.up(x), approximate="tanh")
        return self.drop(self.down(h))


class Block(nn.Module):
    """Pre-norm Transformer block: x + Attn(Norm(x)) [+ CrossAttn(Norm(x), memory)] + MLP(Norm(x))."""

    def __init__(self, cfg: ModelConfig, style: str, n_kv_head: int, cross: bool = False):
        super().__init__()
        gpt2 = style == "gpt2"
        Norm = LayerNorm if gpt2 else RMSNorm
        self.n1 = Norm(cfg.d_model, cfg.norm_eps)
        self.attn = Attention(cfg, n_kv_head, bias=gpt2, use_rope=not gpt2)
        self.has_cross = cross
        if cross:
            self.nc = Norm(cfg.d_model, cfg.norm_eps)
            self.xattn = Attention(cfg, n_kv_head, bias=False, use_rope=False, cross=True)
        self.n2 = Norm(cfg.d_model, cfg.norm_eps)
        self.mlp = MLP(cfg, "gelu" if gpt2 else "swiglu", bias=gpt2)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, rope=None, pos=0, causal=True, cache=None, memory=None, xcache=None):
        x = x + self.drop(self.attn(self.n1(x), rope=rope, pos=pos, causal=causal, cache=cache))
        if self.has_cross:
            x = x + self.drop(self.xattn(self.nc(x), memory=memory, cache=xcache))
        return x + self.mlp(self.n2(x))


# ----------------------------------------------------------------------------------------------------------------
# the language model
# ----------------------------------------------------------------------------------------------------------------
class LegalLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        a = cfg.arch
        assert a in ("gpt2", "llama", "shared", "encdec"), a
        d = cfg.d_model
        if a == "shared":
            self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_embed)
            self.emb_proj = nn.Linear(cfg.d_embed, d, bias=False)          # E -> d   (factorised embedding)
        else:
            self.tok_emb = nn.Embedding(cfg.vocab_size, d)
        if a == "gpt2":
            self.pos_emb = nn.Embedding(cfg.max_seq_len, d)
        self.drop = nn.Dropout(cfg.dropout)

        if a == "gpt2":
            self.blocks = nn.ModuleList([Block(cfg, "gpt2", cfg.n_head) for _ in range(cfg.n_layer)])
        elif a == "llama":
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_layer)])
        elif a == "shared":
            assert cfg.n_layer % cfg.n_unique_layers == 0
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_unique_layers)])
        else:  # encdec
            self.enc_blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_layer)])
            self.enc_norm = RMSNorm(d, cfg.norm_eps)
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head, cross=True) for _ in range(cfg.n_layer)])
        self.norm = LayerNorm(d, cfg.norm_eps) if a == "gpt2" else RMSNorm(d, cfg.norm_eps)

        if a != "gpt2":
            hd = d // cfg.n_head
            cos, sin = rope_cache(hd, max(2 * cfg.max_seq_len, 2048), cfg.rope_theta)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init)
        n_res = 2 * self.n_layer_applications() + (cfg.n_layer if a == "encdec" else 0)
        for name, p in self.named_parameters():
            if name.endswith("o.weight") or name.endswith("down.weight"):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(n_res))

    # -- helpers ------------------------------------------------------------------------------------------------
    def n_layer_applications(self):
        return self.cfg.n_layer * (2 if self.cfg.arch == "encdec" else 1)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def _rope(self):
        return None if self.cfg.arch == "gpt2" else (self.rope_cos, self.rope_sin)

    def _embed(self, idx, pos=0):
        x = self.tok_emb(idx)
        if self.cfg.arch == "shared":
            x = self.emb_proj(x)
        if self.cfg.arch == "gpt2":
            T = idx.shape[1]
            if pos + T > self.cfg.max_seq_len:
                raise ValueError(f"gpt2 positions exceed max_seq_len ({pos + T} > {self.cfg.max_seq_len})")
            x = x + self.pos_emb(torch.arange(pos, pos + T, device=idx.device))[None]
        return self.drop(x)

    def _logits(self, h):
        h = self.norm(h)
        if self.cfg.arch == "shared":
            h = F.linear(h, self.emb_proj.weight.t())     # d -> E   (tied with the input projection)
        return F.linear(h, self.tok_emb.weight)            # tied output embedding

    def _decoder_layers(self):
        if self.cfg.arch == "shared":                       # cycle sharing: 0,1,..,5,0,1,..,5
            return [self.blocks[i % self.cfg.n_unique_layers] for i in range(self.cfg.n_layer)]
        return list(self.blocks)

    def encode(self, src):
        x = self._embed(src)
        for blk in self.enc_blocks:
            x = blk(x, rope=self._rope(), pos=0, causal=False)
        return self.enc_norm(x)

    # -- training forward -----------------------------------------------------------------------------------------
    def forward(self, idx, targets=None, src=None, loss_mask=None):
        """Decoder-only: idx (B,T) -> logits; loss vs targets (B,T).  encdec: src (B,S) is the encoder input."""
        memory = self.encode(src) if self.cfg.arch == "encdec" else None
        x = self._embed(idx)
        for blk in self._decoder_layers():
            x = blk(x, rope=self._rope(), pos=0, causal=True, memory=memory)
        logits = self._logits(x)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), reduction="none")
        if loss_mask is not None:
            m = loss_mask.reshape(-1).float()
            return logits, (loss * m).sum() / m.sum().clamp(min=1)
        return logits, loss.mean()

    # -- generation -----------------------------------------------------------------------------------------------
    @torch.no_grad()
    def generate(self, prompt, max_new_tokens, temperature=1.0, top_k=0, top_p=1.0, repetition_penalty=1.0,
                 no_repeat_ngram=0, banned_ids=(), bos_id=2, generator=None, use_cache=True):
        """prompt: (B, T) LongTensor, all rows the same length (no padding). Returns (B, max_new_tokens) new ids.
        temperature 0 = greedy."""
        B = prompt.shape[0]
        dev = prompt.device
        layers = self._decoder_layers()
        if self.cfg.arch == "encdec":
            memory = self.encode(prompt)
            seq = torch.full((B, 1), bos_id, dtype=torch.long, device=dev)   # decoder starts from <|bos|>
            history = prompt                                                   # for repetition penalties
        else:
            memory, seq, history = None, prompt, prompt
        caches = [dict() for _ in layers] if use_cache else None
        xcaches = [dict() for _ in layers] if (use_cache and memory is not None) else [None] * len(layers)
        out = []
        cur, pos = seq, 0
        for step in range(max_new_tokens):
            if use_cache:
                x = self._embed(cur, pos=pos)
                for li, blk in enumerate(layers):
                    x = blk(x, rope=self._rope(), pos=pos, causal=True, cache=caches[li], memory=memory,
                            xcache=xcaches[li])
                pos += cur.shape[1]
            else:
                x = self._embed(seq)
                for blk in layers:
                    x = blk(x, rope=self._rope(), pos=0, causal=True, memory=memory)
            logits = self._logits(x[:, -1:, :])[:, -1, :].float()
            full_hist = torch.cat([history] + out, dim=1) if out else history
            nxt = sample_next(logits, full_hist, temperature, top_k, top_p, repetition_penalty, no_repeat_ngram,
                              banned_ids, generator)
            out.append(nxt)
            seq = torch.cat([seq, nxt], dim=1)
            cur = nxt
        return torch.cat(out, dim=1)


def sample_next(logits, history, temperature, top_k, top_p, repetition_penalty, no_repeat_ngram, banned_ids,
                generator=None):
    """logits (B,V) float32, history (B,L) long (prompt + generated so far). Returns (B,1)."""
    logits = logits.clone()
    if banned_ids:
        logits[:, list(banned_ids)] = -float("inf")
    if repetition_penalty != 1.0:
        # CTRL-style penalty on tokens already present in the recent history (last 256 tokens)
        recent = history[:, -256:]
        sc = torch.gather(logits, 1, recent)
        sc = torch.where(sc > 0, sc / repetition_penalty, sc * repetition_penalty)
        logits.scatter_(1, recent, sc)
    if no_repeat_ngram and history.shape[1] >= no_repeat_ngram:
        n = no_repeat_ngram
        hist = history[:, -512:].tolist()
        for b, h in enumerate(hist):
            prefix = tuple(h[-(n - 1):]) if n > 1 else tuple()
            banned = {tuple(h[i:i + n])[-1] for i in range(len(h) - n + 1) if tuple(h[i:i + n - 1]) == prefix}
            if banned:
                logits[b, list(banned)] = -float("inf")
    if temperature <= 0:
        return logits.argmax(dim=-1, keepdim=True)
    logits = logits / temperature
    if top_k and top_k > 0:
        kth = torch.topk(logits, min(top_k, logits.shape[-1]), dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p < 1.0:
        sl, si = torch.sort(logits, descending=True, dim=-1)
        cp = torch.softmax(sl, dim=-1).cumsum(dim=-1)
        remove = cp - torch.softmax(sl, dim=-1) > top_p          # keep the smallest set with mass >= top_p
        sl = sl.masked_fill(remove, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, si, sl)
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, 1, generator=generator)

Overwriting /kaggle/working/model.py


In [4]:
%%writefile /kaggle/working/infer.py
"""CS-781 Phase 1 inference: load a checkpoint, generate 128-word continuations, score them with the official
Phase-1 metrics (BERTScore F1 roberta-large / ROUGE-L / BLEU-4), and write a Codabench submission."""
import json, math, os, sys, time, zipfile
from collections import Counter
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from model import LegalLM, ModelConfig

SPECIAL_IDS = list(range(16))            # <|endoftext|>, <|pad|>, ..., <|reserved_9|> - never generated


def load_model(path, device):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = ModelConfig(**ck["config"])
    m = LegalLM(cfg)
    m.load_state_dict(ck["model"])
    m.to(device).eval()
    return m, ck


def _clean(text):
    text = text.encode("ascii", "ignore").decode("ascii")          # test data is pure ASCII
    return " ".join(text.split())


@torch.no_grad()
def generate_texts(model, tok, prompts, dec, batch_size=48, max_new=200, target_words=128, seed=0, log=print):
    """prompts: list[str]. dec: dict(temperature, top_k, top_p, repetition_penalty, no_repeat_ngram).
    Batches prompts of similar token length; inside a batch every prompt is left-truncated to the batch's shortest
    length (a few far-away tokens at most), so no padding is ever needed."""
    device = next(model.parameters()).device
    cuda = device.type == "cuda"
    is_encdec = model.cfg.arch == "encdec"
    max_prompt = 896 if is_encdec else model.cfg.max_seq_len - max_new
    ids = [tok.encode(p, add_special_tokens=False).ids[-max_prompt:] for p in prompts]
    gen_ids = [[] for _ in prompts]
    g = torch.Generator(device=device).manual_seed(seed)
    amp = torch.autocast("cuda", dtype=torch.float16) if cuda else torch.autocast("cpu", enabled=False)
    todo = list(range(len(prompts)))
    for attempt in range(5):                       # extra passes only for continuations still under 128 words
        if not todo:
            break
        order = sorted(todo, key=lambda i: len(ids[i]) + len(gen_ids[i]))
        t0 = time.time()
        for bi in range(0, len(order), batch_size):
            b = order[bi: bi + batch_size]
            ctx = [(ids[i] + gen_ids[i])[-max_prompt:] for i in b] if not is_encdec else [ids[i] for i in b]
            L = min(len(c) for c in ctx)
            x = torch.tensor([c[-L:] for c in ctx], dtype=torch.long, device=device)
            with amp:
                if is_encdec and attempt > 0:      # encdec: re-decode from scratch with a longer budget
                    out = model.generate(x, max_new * (attempt + 1), banned_ids=SPECIAL_IDS, generator=g, **dec)
                else:
                    out = model.generate(x, max_new, banned_ids=SPECIAL_IDS, generator=g, **dec)
            for i, row in zip(b, out.tolist()):
                gen_ids[i] = row if is_encdec else gen_ids[i] + row
        texts = [_clean(tok.decode(gen_ids[i])) for i in range(len(prompts))]
        todo = [i for i, t in enumerate(texts) if len(t.split()) < target_words]
        log(f"   pass {attempt + 1}: {len(order)} prompts in {time.time() - t0:.0f}s; still short: {len(todo)}")
    out, padded = [], 0
    for p, t in zip(prompts, texts):
        w = t.split()[:target_words]
        if len(w) < target_words:                  # last resort (should never trigger with a trained model)
            padded += 1
            w = w + p.split()[-(target_words - len(w)):]
        out.append(" ".join(w))
    if padded:
        log(f"   WARNING: {padded} continuations were shorter than {target_words} words and were padded")
    return out


# ------------------------------------------------------------------------------------------------------------
# Official Phase-1 metrics (same code as evaluate_phase1.py)
# ------------------------------------------------------------------------------------------------------------
def lcs_length(a, b):
    if not a or not b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    masks = {}
    for j, t in enumerate(a):
        masks[t] = masks.get(t, 0) | (1 << j)
    state = 0
    for t in b:
        m = masks.get(t, 0); x = m | state; y = (state << 1) | 1; state = x & ~(x - y)
    return bin(state).count("1")


def rouge_l_f1(p, r):
    pt, rt = p.split(), r.split()
    if not pt or not rt:
        return 0.0
    l = lcs_length(pt, rt)
    if not l:
        return 0.0
    pp, rr = l / len(pt), l / len(rt)
    return 0.0 if pp + rr == 0 else 2 * pp * rr / (pp + rr)


def ngrams(t, n):
    return Counter(tuple(t[i:i + n]) for i in range(len(t) - n + 1)) if len(t) >= n else Counter()


def bleu4(preds, refs):
    order = 4; clipped = [0] * order; totals = [0] * order; pl = rl = 0
    for p, r in zip(preds, refs):
        pl += len(p); rl += len(r)
        for n in range(1, order + 1):
            pn = ngrams(p, n); rn = ngrams(r, n); totals[n - 1] += sum(pn.values())
            for g, c in pn.items():
                clipped[n - 1] += min(c, rn.get(g, 0))
    prec = [clipped[i] / totals[i] if totals[i] else 0.0 for i in range(order)]
    if any(x <= 0 for x in prec) or pl == 0:
        return 0.0
    bp = 1.0 if pl > rl else math.exp(1.0 - rl / pl)
    return bp * math.exp(sum(math.log(x) for x in prec) / order)


def distinct_n(texts, n=2):
    tot = uniq = 0
    for t in texts:
        g = ngrams(t.split(), n); tot += sum(g.values()); uniq += len(g)
    return uniq / max(1, tot)


def score(preds, refs, bert_batch=32):
    """preds/refs: lists of strings in the same order. Returns dict with the official metrics + diagnostics."""
    import bert_score
    device = "cuda" if torch.cuda.is_available() else "cpu"
    _, _, f = bert_score.score([p.strip() for p in preds], [r.strip() for r in refs], lang="en",
                               batch_size=bert_batch, verbose=False, device=device)
    return {"bertscore_f1": float(f.mean().item()),
            "rouge_l": sum(rouge_l_f1(p, r) for p, r in zip(preds, refs)) / len(refs),
            "bleu4": bleu4([p.split() for p in preds], [r.split() for r in refs]),
            "distinct2": distinct_n(preds, 2),
            "avg_words": sum(len(p.split()) for p in preds) / len(preds)}


def write_submission(pred_by_id, test_ids, out_dir):
    """Validates exactly like the official evaluator, writes prediction.json and submission.zip (file at zip root)."""
    assert set(pred_by_id) == set(test_ids), "prediction ids must match test ids exactly"
    for k, v in pred_by_id.items():
        assert isinstance(v, str) and v.strip(), f"empty prediction for id {k}"
        assert len(v.split()) == 128, f"id {k} has {len(v.split())} words"
    os.makedirs(out_dir, exist_ok=True)
    pj = os.path.join(out_dir, "prediction.json")
    with open(pj, "w", encoding="utf-8") as f:
        json.dump({k: {"text": pred_by_id[k]} for k in test_ids}, f, ensure_ascii=False, indent=2)
    zp = os.path.join(out_dir, "submission.zip")
    with zipfile.ZipFile(zp, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"]
    back = json.load(open(pj, encoding="utf-8"))
    assert list(back) == list(test_ids)
    return pj, zp

Writing /kaggle/working/infer.py


cell 4

In [5]:
# ===================== NB3 CELL 4 - Helpers + calibration baselines =====================
sys.path.insert(0, WORK)
import importlib, infer; importlib.reload(infer)
from infer import load_model, generate_texts, score, write_submission

def get_ckpt(run, fname):
    """Download runs/<run>/<fname> from our private checkpoint repo (cached)."""
    return hf_hub_download(CKPT_REPO, f"runs/{run}/{fname}", repo_type="model", token=HF_TOKEN, local_dir=CKPT_DIR)

def eval_dev(model, dec, n=None, batch_size=48, seed=0):
    ids = DEV_IDS[:n] if n else DEV_IDS
    t0 = time.time()
    preds = generate_texts(model, tok, [dev_prompts[i]["text"] for i in ids], dec, batch_size=batch_size, seed=seed)
    gen_s = time.time() - t0
    res = score(preds, [dev_truth[i]["text"] for i in ids])
    res["gen_seconds"] = round(gen_s, 1)
    return res, dict(zip(ids, preds))

def show(name, res):
    print(f"{name:42s} BERTScore {res['bertscore_f1']:.4f} | ROUGE-L {res['rouge_l']:.4f} | BLEU-4 {res['bleu4']:.4f}"
          f" | distinct-2 {res['distinct2']:.3f} | words {res['avg_words']:.0f}"
          + (f" | gen {res['gen_seconds']:.0f}s" if 'gen_seconds' in res else ""), flush=True)

# Calibration: what do trivial "continuations" score? (tells us where a real model must land)
refs = [dev_truth[i]["text"] for i in DEV_IDS]
copy_tail = [" ".join(dev_prompts[i]["text"].split()[-128:]) for i in DEV_IDS]      # last 128 prompt words
shuffled = refs[1:] + refs[:1]                                                       # a DIFFERENT doc's reference
CALIB = {"baseline: copy last 128 prompt words": score(copy_tail, refs),
         "baseline: another document's reference": score(shuffled, refs)}
for k, v in CALIB.items():
    show(k, v)
json.dump(CALIB, open(f"{RESULTS}/calibration.json", "w"), indent=2)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

... (2 outputs left out)

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


baseline: copy last 128 prompt words       BERTScore 0.8424 | ROUGE-L 0.1979 | BLEU-4 0.0782 | distinct-2 0.910 | words 128
baseline: another document's reference     BERTScore 0.8107 | ROUGE-L 0.1551 | BLEU-4 0.0099 | distinct-2 0.912 | words 128


# Cell 10 for 100M token submission

In [6]:
# ===================== NB3 CELL 10 - Test submissions from the 4 exploration models (100M tokens each) =====================
# One submission per ARCHITECTURE (a different trained model each time, not a hyperparameter change).
# Every model uses the SAME decoding as Submission 1 (T0.5, top-p 0.9, rep-penalty 1.1, seed 0), so the
# leaderboard numbers compare the architectures fairly. Needs only Cells 1-4 to have been run.
EXPLORE_RUNS = ["explore_gpt2", "explore_shared", "explore_encdec", "explore_llama"]   # last one optional
EXPLORE_FILE = "final_model.pt"                     # the same checkpoints that were compared in Cell 5
DEC_FAIR     = dict(temperature=0.5, top_p=0.9, repetition_penalty=1.1)

ZIPS = f"{WORK}/zips"; os.makedirs(ZIPS, exist_ok=True)
prompts_test = [test_data[i]["text"] for i in TEST_IDS]
arch_subs = {}
for run in EXPLORE_RUNS:
    print(f"\n========== {run} ==========", flush=True)
    t_run = time.time()
    try:
        model, ck = load_model(get_ckpt(run, EXPLORE_FILE), DEVICE)
        info = dict(run=run, file=EXPLORE_FILE, arch=model.cfg.arch, params_M=round(model.num_params() / 1e6, 2),
                    tokens_seen=ck["tokens_seen"], step=ck["step"], dev_loss=ck["metrics"].get("dev_loss"),
                    dev_ppl=ck["metrics"].get("dev_ppl"), decoding=DEC_FAIR, seed=0)
        print(f"loaded: arch {info['arch']} | {info['params_M']}M params | {info['tokens_seen']/1e6:.0f}M tokens "
              f"| dev loss {info['dev_loss']}", flush=True)

        # 1) dev set (479 held-out SC docs) with the official metrics -> numbers for the report
        dev_res, _ = eval_dev(model, DEC_FAIR, seed=0)
        info["dev_metrics"] = dev_res
        show(f"{run} DEV", dev_res)

        # 2) the 1000 test prompts -> submission.zip
        t0 = time.time()
        preds = generate_texts(model, tok, prompts_test, DEC_FAIR, batch_size=48, seed=0)
        info["test_gen_seconds"] = round(time.time() - t0)
        sub_dir = f"{WORK}/submission_{run}"
        pj, zp = write_submission(dict(zip(TEST_IDS, preds)), TEST_IDS, sub_dir)
        json.dump(info, open(f"{sub_dir}/submission_info.json", "w"), indent=2)
        shutil.copy(zp, f"{ZIPS}/{run}_submission.zip")     # easy to find in the Output panel
        tag = f"{run}_{EXPLORE_FILE.split('.')[0]}_T0.5_p0.9_rep1.1"
        for f in ["prediction.json", "submission.zip", "submission_info.json"]:
            api.upload_file(path_or_fileobj=f"{sub_dir}/{f}", path_in_repo=f"submissions/{tag}/{f}",
                            repo_id=CKPT_REPO, repo_type="model")
        arch_subs[run] = info
        print(f"READY: {ZIPS}/{run}_submission.zip  (test generation {info['test_gen_seconds']}s, "
              f"run total {time.time()-t_run:.0f}s)", flush=True)
        print("   example:", preds[0][:250])
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f"!!! {run} FAILED: {type(e).__name__}: {e}  -> continuing with the next model", flush=True)
    finally:
        model = None; torch.cuda.empty_cache()

json.dump(arch_subs, open(f"{RESULTS}/architecture_submissions.json", "w"), indent=2)
api.upload_file(path_or_fileobj=f"{RESULTS}/architecture_submissions.json",
                path_in_repo="results/architecture_submissions.json", repo_id=CKPT_REPO, repo_type="model")

print("\n" + "=" * 100)
print(f"{'run':16s} {'arch':8s} {'params':>7s} {'tokens':>7s} {'dev loss':>8s} {'dev BERTScore':>13s} "
      f"{'ROUGE-L':>8s} {'BLEU-4':>7s}  zip")
for run, i in arch_subs.items():
    d = i["dev_metrics"]
    print(f"{run:16s} {i['arch']:8s} {i['params_M']:6.1f}M {i['tokens_seen']/1e6:6.0f}M {i['dev_loss']:8.3f} "
          f"{d['bertscore_f1']:13.4f} {d['rouge_l']:8.4f} {d['bleu4']:7.4f}  zips/{run}_submission.zip")
print("Missing:", [r for r in EXPLORE_RUNS if r not in arch_subs] or "none")


========== explore_gpt2 ==========


runs/explore_gpt2/final_model.pt: reconstructing file:   0%|          |  0.00B /  444MB            

runs/explore_gpt2/final_model.pt: downloading bytes:           |  0.00B            

loaded: arch gpt2 | 111.01M params | 100M tokens | dev loss 3.2698313369740672
   pass 1: 479 prompts in 49s; still short: 46
   pass 2: 46 prompts in 5s; still short: 7
   pass 3: 7 prompts in 2s; still short: 6
   pass 4: 6 prompts in 2s; still short: 5
   pass 5: 5 prompts in 2s; still short: 5


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (27 outputs left out)

explore_llama DEV                          BERTScore 0.8272 | ROUGE-L 0.1832 | BLEU-4 0.0365 | distinct-2 0.805 | words 128 | gen 80s
   pass 1: 1000 prompts in 133s; still short: 97
   pass 2: 97 prompts in 19s; still short: 17
   pass 3: 17 prompts in 4s; still short: 11
   pass 4: 11 prompts in 4s; still short: 10
   pass 5: 10 prompts in 3s; still short: 9


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

READY: /kaggle/working/zips/explore_llama_submission.zip  (test generation 164s, run total 292s)
   example: a matter of common knowledge that the impugned notification is vitiated because the land acquisition proceedings are in violation of Article 14 of the Constitution of India. The question whether the land acquisition proceedings are invalid or not is 

run              arch      params  tokens dev loss dev BERTScore  ROUGE-L  BLEU-4  zip
explore_gpt2     gpt2      111.0M    100M    3.270        0.8266   0.1875  0.0362  zips/explore_gpt2_submission.zip
explore_shared   shared     58.7M    100M    3.759        0.8258   0.1863  0.0284  zips/explore_shared_submission.zip
explore_encdec   encdec    117.2M    100M    3.582        0.8210   0.1781  0.0217  zips/explore_encdec_submission.zip
explore_llama    llama     107.8M    100M    2.962        0.8272   0.1832  0.0365  zips/explore_llama_submission.zip
Missing: none
